# 05 — Inference Benchmark (with Test-Set Evaluation)

Evaluate the fine-tuned model end-to-end using **vLLM** on a held-out
test split:

1. Load test Q&A pairs from the StackSample dataset.
2. Generate responses with vLLM (PagedAttention + continuous batching).
3. Compute **ROUGE-1/2/L** and **BLEU** against ground-truth answers.
4. Report **throughput**, **latency**, **CPU**, **memory**, and **GPU** usage.

In [ ]:
import sys
sys.path.append("..")

from pathlib import Path
import json
import time
import gc
import torch
import polars as pl
import psutil

from src.llm_optimization.inference import VLLMEngine, prepare_for_vllm
from src.llm_optimization.core import InferenceConfig
from src.llm_optimization.evaluation import (
    EvaluationReport,
    compute_rouge_scores,
    compute_bleu_score,
)

METHODS = [
    ("qlora",                 "QLoRA"),
    ("mixed_precision",       "Mixed Precision"),
    ("gradient_checkpointing","Grad Checkpointing"),
    ("prompt_tuning",         "Prompt Tuning"),
    ("knowledge_distillation","Distillation"),
    # ("zero3", "ZeRO-3"),    # uncomment if you ran it
]

# Same test set for every method (fair comparison)
_, test_df, _ = load_and_prepare_data(DataConfig(dataset_path='./data', max_samples=200))
test_df = test_df.head(100)

prompts, references, questions = [], [], []
for row in test_df.iter_rows(named=True):
    prompts.append(f'Question: {row["question_title"]}\n{row["question_body"]}\nAnswer:')
    references.append(row["answer"])
    questions.append(row["question_title"])

all_reports = {}

for slug, display_name in METHODS:
    artifact = Path(f'./outputs/{slug}')
    if not artifact.exists():
        print(f' Skipping {display_name}: {artifact} not found')
        continue

    print(f'\n{"=" * 70}\nEvaluating: {display_name}\n{"=" * 70}')

    # Merge PEFT if needed
    try:
        model_path = prepare_for_vllm(artifact)
    except Exception as e:
        print(f' Could not prepare {slug}: {e}')
        continue

    # Fresh vLLM engine per method
    engine = VLLMEngine(InferenceConfig(
        model_path=str(model_path),
        max_new_tokens=128,
        temperature=0.0,
        enforce_eager=True,
    ))

    report = EvaluationReport(method=display_name)

    BATCH = 16
    for i in range(0, len(prompts), BATCH):
        batch_p = prompts[i:i + BATCH]
        batch_r = references[i:i + BATCH]
        batch_q = questions[i:i + BATCH]

        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        t0 = time.time()
        responses = engine.generate(batch_p)
        elapsed = time.time() - t0
        per_sample = elapsed / len(batch_p)

        cpu = psutil.cpu_percent(interval=None)
        mem = psutil.virtual_memory().percent
        gpu_mb = torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else 0.0

        for q, ref, resp in zip(batch_q, batch_r, responses):
            r = compute_rouge_scores(ref, resp)
            report.add_sample(
                question=q, ground_truth=ref, response=resp,
                num_tokens=len(resp.split()), latency=per_sample,
                cpu_percent=cpu, memory_percent=mem, gpu_memory_mb=gpu_mb,
                rouge_scores=r, bleu_score=compute_bleu_score(ref, resp),
            )

    report.print_report()
    all_reports[display_name] = report.summary()

    # Free VRAM before the next method
    del engine
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Comparison table
import pandas as pd

comparison = pd.DataFrame([
    {
        "Method": name,
        "ROUGE-1": s["rouge1_f1"],
        "ROUGE-2": s["rouge2_f1"],
        "ROUGE-L": s["rougeL_f1"],
        "BLEU":    s["bleu_score"],
        "Latency (ms)": s["avg_latency_seconds"] * 1000,
        "Throughput (tok/s)": s["avg_throughput_tokens_per_sec"],
        "GPU mem (MB)": s["avg_gpu_memory_mb"],
    }
    for name, s in all_reports.items()
]).round(4)

print("\n" + "=" * 100)
print("CROSS-METHOD COMPARISON")
print("=" * 100)
print(comparison.to_string(index=False))

Path('./outputs').mkdir(exist_ok=True)
comparison.to_csv('./outputs/inference_comparison.csv', index=False)
with open('./outputs/inference_comparison.json', 'w') as f:
    json.dump(all_reports, f, indent=2)

print("\nReport saved")